In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import osmnx as ox
from pathlib import Path
from datetime import timedelta
from numba import jit
from tqdm import tqdm

# =============================================================
# BALANCE MAIN MODEL
# =============================================================

MASTER_SEED = 42
np.random.seed(MASTER_SEED)

# ----- scenario (fixed to HIGH for all model comparisons) ---
SCENARIO = {"delay_factor": 1.5, "lambda": 3.0}

# ----- vehicle service times --------------------------------
SERVICE_TIME = {"motorcycle": 10, "truck": 20}   # minutes

# ----- travel noise ----------------------------------------
TRAVEL_NOISE_RANGE = (-0.10, 0.10)               # ±10 %

# ----- weather ---------------------------------------------
WEATHER_FACTORS   = {"clear": 1.0, "rain": 0.85, "heavy_rain": 0.75}
WEATHER_PROBS     = [0.60, 0.30, 0.10]           # clear / rain / heavy_rain
WEATHER_BLOCK_HRS = (1.0, 3.0)                   # each block lasts 1–3 h

# ----- radius expansion levels (km) -----------------------
RADIUS_LEVELS = [3, 5, 8, 12]                    # tried in order

# ----- other routing constants ----------------------------
MAX_ROUTE_DISTANCE     = 100                      # km hard cap
TRUCK_CENTRAL_LOCATION = (10.775, 106.680)
TRUCK_SHIPPERS = [
    "65a4bd8c12c52b180d74a899",
    "645dc07b435f2884d83fe0d0",
    "6407e82bfbb0d0000624ffd2",
]

# ----- time-of-day availability probabilities -------------
def tod_availability_prob(hour: int) -> float:
    if   6  <= hour <  9:  return np.random.uniform(0.40, 0.60)
    elif 9  <= hour < 11:  return np.random.uniform(0.60, 0.75)
    elif 11 <= hour < 13:  return np.random.uniform(0.70, 0.85)
    elif 13 <= hour < 15:  return np.random.uniform(0.55, 0.70)
    elif 15 <= hour < 18:  return np.random.uniform(0.60, 0.75)
    elif 18 <= hour < 21:  return np.random.uniform(0.65, 0.85)
    elif 21 <= hour < 23:  return np.random.uniform(0.40, 0.60)
    else:                  return np.random.uniform(0.15, 0.35)

# =============================================================
#  DATA LOADING
#  Balance loads orders_test_SHARED.csv produced by the greedy
#  script so noise, weather, hour_bucket are byte-for-byte
#  identical — guaranteed fair comparison.
# =============================================================

print("📊  Loading shared test environment …")

SHARED_PATH = Path.home() / "Downloads/orders_test_SHARED.csv"

if SHARED_PATH.exists():
    # ── preferred path: reuse greedy's pre-generated columns ──
    orders_test = pd.read_csv(SHARED_PATH)
    orders_test["createdAt"]            = pd.to_datetime(orders_test["createdAt"])
    orders_test["expectedDeliveryTime"] = pd.to_datetime(orders_test["expectedDeliveryTime"])
    orders_test["hour_bucket"]          = pd.to_datetime(orders_test["hour_bucket"])
    orders_test = orders_test.sort_values("createdAt").reset_index(drop=True)
    print(f"   Loaded {len(orders_test)} orders from shared file ✓")
    _shared_loaded = True

else:
    # ── fallback: regenerate with the same seeds (run greedy first!) ──
    print("   ⚠️  Shared file not found — regenerating with same seeds.")
    print("   Run dispatch_greedy_fixed.py first to produce orders_test_SHARED.csv")

    df = pd.read_csv(Path.home() / "Downloads/synthetic_orders_FINAL_CORRECT.csv")
    df["createdAt"]            = pd.to_datetime(df["createdAt"])
    df["expectedDeliveryTime"] = pd.to_datetime(df["expectedDeliveryTime"])
    df = df.sort_values("createdAt").reset_index(drop=True)

    start     = df["createdAt"].min()
    train_end = start     + pd.Timedelta(days=154)
    val_end   = train_end + pd.Timedelta(days=44)
    orders_test = df[df["createdAt"] >= val_end].sort_values("createdAt").reset_index(drop=True)

    # noise
    orders_test["noise"] = np.random.uniform(
        TRAVEL_NOISE_RANGE[0], TRAVEL_NOISE_RANGE[1], size=len(orders_test)
    )

    # weather time-series
    def build_weather_schedule(orders_df, seed=MASTER_SEED):
        rng   = np.random.default_rng(seed)
        t_min = orders_df["createdAt"].min()
        t_max = orders_df["createdAt"].max()
        schedule = []
        cur = t_min
        while cur <= t_max:
            dur     = pd.Timedelta(hours=float(rng.uniform(*WEATHER_BLOCK_HRS)))
            weather = rng.choice(["clear", "rain", "heavy_rain"], p=WEATHER_PROBS)
            schedule.append((cur, cur + dur, weather))
            cur += dur
        return schedule

    def lookup_weather(ts, schedule):
        for blk_start, blk_end, w in schedule:
            if blk_start <= ts < blk_end:
                return w
        return "clear"

    weather_schedule = build_weather_schedule(orders_test)
    orders_test["weather"] = orders_test["createdAt"].apply(
        lambda ts: lookup_weather(ts, weather_schedule)
    )
    orders_test["hour_bucket"] = orders_test["createdAt"].dt.floor("h")
    _shared_loaded = False

    print(f"   Regenerated {len(orders_test)} test orders")

# =============================================================
#  SHIPPER LOCATIONS  (fixed; same as greedy)
# =============================================================

SHIPPER_LOCATIONS = {
    "6407e82bfbb0d0000624ffd1": (10.7662, 106.6759),
    "6407e82bfbb0d0000624ffd2": (10.8261, 106.7064),
    "6416bcf7349718f9506caee9": (10.8339, 106.6772),
    "641aa31f10ea5e9b8db76e1a": (10.8155, 106.7250),
    "6423b7b8e008138acf0fab39": (10.7605, 106.7040),
    "645dc07b435f2884d83fe0d0": (10.7329, 106.6784),
    "64b07c61789e336ea4fa9986": (10.8025, 106.6653),
    "64bb4c2362f9b15cf431583b": (10.7248, 106.6969),
    "64bb63c662f9b15cf4315ae6": (10.7870, 106.6761),
    "65a11cd812c52b180d7499d2": (10.7398, 106.6400),
    "65a4bd8c12c52b180d74a899": (10.8016, 106.6605),
    "65d2db75347c9179ee9bf18c": (10.7850, 106.6667),   # typo fixed (was …9bf18)
    "65dd9522e167fc5315486c19": (10.7867, 106.7057),
    "65ec1239c95b44438437e9ae": (10.8055, 106.6941),
    "65f3c1c03c2e30e7ee32eded": (10.8450, 106.6667),
    "663da005ecf5fd58bbe6fc8a": (10.8129, 106.7131),
}
shipper_ids    = list(SHIPPER_LOCATIONS.keys())
shipper_coords = np.array([SHIPPER_LOCATIONS[s] for s in shipper_ids])

# =============================================================
#  AVAILABILITY MAP  — built from the same seed as greedy
#  Keyed by hour-bucket so all orders in the same hour share
#  the same online set (consistent with hourly login shifts).
# =============================================================

unique_buckets = orders_test["hour_bucket"].unique()
rng_avail = np.random.default_rng(MASTER_SEED + 1)   # same secondary seed as greedy

availability_map: dict = {}
for bucket in unique_buckets:
    prob   = tod_availability_prob(pd.Timestamp(bucket).hour)
    online = [s for s in shipper_ids if rng_avail.random() < prob]
    if not online:
        online = [rng_avail.choice(shipper_ids)]
    availability_map[pd.Timestamp(bucket)] = online

def get_online_shippers(ts: pd.Timestamp) -> list:
    return availability_map[ts.floor("h")]

# =============================================================
#  GRAPH & NODES
# =============================================================

print("🗺  Loading road graph …")
G   = ox.load_graphml("hcmc_wards_drive.graphml")
G_u = G.to_undirected()

shipper_nodes = {
    sid: ox.distance.nearest_nodes(G, lon, lat)
    for sid, (lat, lon) in SHIPPER_LOCATIONS.items()
}
truck_depot_node = ox.distance.nearest_nodes(
    G, TRUCK_CENTRAL_LOCATION[1], TRUCK_CENTRAL_LOCATION[0]
)

# =============================================================
#  SHARED UTILITY FUNCTIONS  — identical to greedy
# =============================================================

@jit(nopython=True)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = (np.sin(dlat / 2) ** 2
         + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2))
         * np.sin(dlon / 2) ** 2)
    return 2.0 * R * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))


def get_speed(vehicle: str, dt: pd.Timestamp) -> float:
    h    = dt.hour
    base = 30 if vehicle == "motorcycle" else 20
    if   h in [7, 8, 9, 17, 18, 19]: return base * 0.70
    elif h in [11, 12, 13]:           return base * 0.80
    else:                             return float(base)


def estimate_travel_time(
    distance_km: float,
    dt: pd.Timestamp,
    vehicle: str,
    weather: str,
    noise: float,
    delay_factor: float,
) -> float:
    speed = get_speed(vehicle, dt)
    base  = (distance_km / speed) * 60.0
    return base * delay_factor * WEATHER_FACTORS[weather] * (1.0 + noise)


def get_vehicle(row) -> str:
    return "truck" if "ban_tai" in str(row.serviceType).lower() else "motorcycle"


distance_cache: dict = {}

def road_distance(n1: int, n2: int) -> float:
    key = (min(n1, n2), max(n1, n2))
    if key in distance_cache:
        return distance_cache[key]
    try:
        d = nx.shortest_path_length(G_u, n1, n2, weight="length") / 1000.0
        d = min(d, MAX_ROUTE_DISTANCE)
    except Exception:
        d = MAX_ROUTE_DISTANCE
    distance_cache[key] = d
    return d


def get_candidates(
    order_lat: float,
    order_lon: float,
    online: list,
    vehicle: str,
) -> list:
    """
    Expanding-radius candidate filter — identical logic to greedy.
    3 → 5 → 8 → 12 km.  Fallback: nearest 3 from online pool.
    Truck orders are restricted to TRUCK_SHIPPERS.
    """
    pool = [s for s in online if vehicle != "truck" or s in TRUCK_SHIPPERS]
    if not pool:
        return []

    dists = {s: haversine(order_lat, order_lon, *SHIPPER_LOCATIONS[s]) for s in pool}

    for radius in RADIUS_LEVELS:
        candidates = [s for s, d in dists.items() if d <= radius]
        if candidates:
            return candidates

    return sorted(dists, key=dists.get)[:3]   # absolute fallback


# =============================================================
#  BALANCE DISPATCH  (model-specific logic)
#
#  Objective:  metric = cumulative_workload + cost
#              cost   = (d1 + d2) + λ·lateness + pickup_penalty
#
#  This differs from greedy (cost only) by adding workload —
#  it spreads deliveries across shippers rather than always
#  picking the locally cheapest one.
# =============================================================

print("\n⚖️   Running BALANCE dispatch (HIGH scenario, test set) …")

DELAY = SCENARIO["delay_factor"]
LAM   = SCENARIO["lambda"]

# ----- initialise shipper states ---------------------------
shipper_state = {
    sid: {
        "last_delivery": pd.Timestamp("2025-05-07 07:00:00"),
        "current_node":  shipper_nodes[sid],
        "workload":      0.0,          # ← cumulative km; used in balance metric
    }
    for sid in shipper_ids
}
truck_state = {
    sid: {"available_time": pd.Timestamp("2025-05-07 07:00:00")}
    for sid in TRUCK_SHIPPERS
}

results = []

for order in tqdm(orders_test.itertuples(), total=len(orders_test)):

    vehicle = get_vehicle(order)
    noise   = order.noise
    weather = order.weather
    online  = get_online_shippers(order.createdAt)

    pu = ox.distance.nearest_nodes(G, order.senderLng,   order.senderLat)
    dr = ox.distance.nearest_nodes(G, order.receiverLng, order.receiverLat)

    candidates = get_candidates(order.senderLat, order.senderLng, online, vehicle)

    if not candidates:
        # Zero truck shippers online — truly unassignable
        continue

    # --- truck: keep only those not currently busy ----------
    if vehicle == "truck":
        free = [
            s for s in candidates
            if order.createdAt >= truck_state[s]["available_time"]
        ]
        if not free:
            # All busy — take the one who frees up soonest
            free = [min(TRUCK_SHIPPERS, key=lambda s: truck_state[s]["available_time"])]
        candidates = free

    # --- balance metric minimisation -----------------------
    best_metric  = np.inf
    best_shipper = None
    best_details = None

    for sid in candidates:
        st         = shipper_state[sid]
        start_time = max(order.createdAt, st["last_delivery"])

        d1 = road_distance(st["current_node"], pu)
        d2 = road_distance(pu, dr)

        if d1 + d2 > MAX_ROUTE_DISTANCE:
            continue

        t1       = estimate_travel_time(d1, start_time, vehicle, weather, noise, DELAY)
        pickup_t = start_time + timedelta(minutes=t1)
        t2       = estimate_travel_time(d2, pickup_t,   vehicle, weather, noise, DELAY)
        service  = SERVICE_TIME[vehicle]
        completion = pickup_t + timedelta(minutes=(t2 + service))

        lateness = max(0.0, (completion - order.expectedDeliveryTime).total_seconds() / 60.0)

        # pickup_penalty: extra cost for shippers found outside the 12 km radius
        # (d1 > RADIUS_LEVELS[-1] only when fallback kicked in)
        pickup_penalty = max(0.0, d1 - RADIUS_LEVELS[-1]) * 3.0

        cost   = (d1 + d2) + LAM * lateness + pickup_penalty

        # ── balance objective: penalise already-busy shippers ──
        metric = st["workload"] + cost

        if metric < best_metric:
            best_metric  = metric
            best_shipper = sid
            best_details = (d1, d2, lateness, completion)

    if best_shipper is None:
        # Every candidate exceeded MAX_ROUTE_DISTANCE — assign nearest anyway
        best_shipper = candidates[0]
        st           = shipper_state[best_shipper]
        start_time   = max(order.createdAt, st["last_delivery"])
        d1 = road_distance(st["current_node"], pu)
        d2 = road_distance(pu, dr)
        t1       = estimate_travel_time(d1, start_time, vehicle, weather, noise, DELAY)
        pickup_t = start_time + timedelta(minutes=t1)
        t2       = estimate_travel_time(d2, pickup_t,   vehicle, weather, noise, DELAY)
        service  = SERVICE_TIME[vehicle]
        completion = pickup_t + timedelta(minutes=(t2 + service))
        lateness   = max(0.0, (completion - order.expectedDeliveryTime).total_seconds() / 60.0)
        best_details = (d1, d2, lateness, completion)

    d1, d2, lateness, completion = best_details
    total_dist = d1 + d2

    # --- truck return to depot ------------------------------
    if vehicle == "truck":
        return_dist = road_distance(dr, truck_depot_node)
        t_r         = estimate_travel_time(return_dist, completion, "truck", weather, noise, DELAY)
        completion += timedelta(minutes=t_r)
        total_dist += return_dist

        shipper_state[best_shipper]["current_node"]   = truck_depot_node
        truck_state[best_shipper]["available_time"]   = completion
    else:
        shipper_state[best_shipper]["current_node"] = dr

    shipper_state[best_shipper]["last_delivery"]  = completion
    shipper_state[best_shipper]["workload"]       += total_dist   # accumulate for balance

    system_time = (completion - order.createdAt).total_seconds() / 60.0

    results.append({
        "order":           order.id,
        "shipper":         best_shipper,
        "vehicle":         vehicle,
        "weather":         weather,
        "noise":           round(noise, 4),
        "distance_km":     round(total_dist, 2),
        "lateness_min":    round(lateness, 2),
        "system_time_min": round(system_time, 2),
        "on_time":         int(lateness == 0),
        "method":          "BALANCE",
        "scenario":        "high",
    })

# =============================================================
#  SAVE
# =============================================================

out = Path.home() / "Downloads/dispatch_BALANCE_FIXED.csv"
pd.DataFrame(results).to_csv(out, index=False)
print(f"\n✅  Saved {len(results)} rows → {out}")

# =============================================================
#  SUMMARY
# =============================================================

df_out = pd.DataFrame(results)
print("\n📊  BALANCE SUMMARY (test set, HIGH scenario):")
print(df_out.agg({
    "order":           "count",
    "distance_km":     ["mean", "std"],
    "lateness_min":    ["mean", "std"],
    "system_time_min": ["mean", "std"],
    "on_time":         "mean",
}).round(2))

print("\n🌦  Weather breakdown:")
print(df_out.groupby("weather")[["lateness_min", "on_time"]].mean().round(3))

print("\n👷  Workload distribution across shippers:")
wl = pd.Series({sid: shipper_state[sid]["workload"] for sid in shipper_ids})
print(wl.describe().round(2))
print(f"   Gini coefficient (0=perfect balance): {((wl.sort_values().values * (2*np.arange(1,len(wl)+1) - len(wl) - 1)).sum()) / (len(wl) * wl.sum()):.4f}")
